# RAG on Azure — Day 3: Q&A Assistant with Robust "I Don't Know" 

**Use case:** a support/FAQ bot that answers when it can, and clearly refuses when the answer isn't in the documents — instead of hallucinating.

- A **two-layer defense** against hallucination:
  1. *Score threshold gate* — if retrieval is too weak, we refuse **before** calling the LLM at all (saves cost and avoids forcing the model to answer with bad context).
  2. *Strict sentinel prompt* — when the LLM is called, it must reply with the exact token `INSUFFICIENT_INFORMATION` if the answer isn't in context. We detect that and surface a clean refusal.
- A small **evaluation set** of in-scope vs out-of-scope questions with a confusion-matrix-style report.
- A **threshold sweep** so you can pick the cutoff using evidence from your own data.


## 0. Install dependencies

In [1]:
%pip install -q "openai>=1.55.3" "azure-search-documents>=11.5.1" python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Load configuration (robust loader)

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

explicit = Path(r"C:\Users\sunil\OneDrive\Documents\GitHub\Basic RAG\RAG\.env")
env_path = explicit if explicit.exists() else (Path(find_dotenv()) if find_dotenv() else None)

if env_path and Path(env_path).exists():
    for line in Path(env_path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        os.environ[k.strip()] = v.strip().strip('"').strip("'")
    load_dotenv(env_path, override=False)
else:
    raise FileNotFoundError("No .env file found.")

AZURE_OPENAI_ENDPOINT    = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY     = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21")
CHAT_DEPLOYMENT          = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT", "gpt-4o")
EMBEDDING_DEPLOYMENT     = os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_SEARCH_ENDPOINT    = os.getenv("AZURE_SEARCH_ENDPOINT")
AZURE_SEARCH_API_KEY     = os.getenv("AZURE_SEARCH_API_KEY")
INDEX_NAME               = os.getenv("AZURE_SEARCH_INDEX", "rag-handbook-day2")

for _name, _val in [
    ("AZURE_OPENAI_ENDPOINT", AZURE_OPENAI_ENDPOINT),
    ("AZURE_OPENAI_API_KEY",  AZURE_OPENAI_API_KEY),
    ("AZURE_SEARCH_ENDPOINT", AZURE_SEARCH_ENDPOINT),
    ("AZURE_SEARCH_API_KEY",  AZURE_SEARCH_API_KEY),
]:
    assert _val, f"Set {_name} in your .env"

print("Config OK. Using index:", INDEX_NAME)

Config OK. Using index: ragpipeline-index


## 2. Clients

In [4]:
from openai import AzureOpenAI
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

aoai = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)
search_cred = AzureKeyCredential(AZURE_SEARCH_API_KEY)
search_client = SearchClient(endpoint=AZURE_SEARCH_ENDPOINT, index_name=INDEX_NAME, credential=search_cred)
print("Clients ready.")

Clients ready.


## 3. Verify that data is populated

In [5]:
count = search_client.get_document_count()
print(f"Index '{INDEX_NAME}' has {count} documents.")
assert count > 0, "Index is empty — run Day 2 to ingest documents first."

Index 'ragpipeline-index' has 10 documents.


## 4. Retrieve (returns scores)

In [6]:
from azure.search.documents.models import VectorizedQuery

def embed(text):
    return aoai.embeddings.create(model=EMBEDDING_DEPLOYMENT, input=[text]).data[0].embedding

def retrieve(query, k=5):
    qvec = embed(query)
    vq = VectorizedQuery(vector=qvec, k_nearest_neighbors=k, fields="contentVector")
    results = search_client.search(search_text=None, vector_queries=[vq], select=["content", "source"])
    return [
        {"content": r["content"], "source": r["source"], "score": r["@search.score"]}
        for r in results
    ]

## 5. The two-layer defense

**Layer 1 — score threshold gate.** If the top retrieved chunk's score is below `SCORE_THRESHOLD`, we don't call the LLM at all. This is the cheap, fast first defense: bad context in, bad answer out, so just refuse.

**Layer 2 — strict sentinel prompt.** When we *do* call the LLM, the system prompt tells it to reply with exactly `INSUFFICIENT_INFORMATION` if the answer isn't in the passages. We detect that token and turn it into a clean refusal.

Each call returns a `status`: `answered` / `refused-low-confidence` / `refused-by-model`, so we can measure later what fired and why.

In [7]:
SCORE_THRESHOLD = 0.65            # tune this from the threshold sweep below
REFUSAL_SENTINEL = "INSUFFICIENT_INFORMATION"

SYSTEM_PROMPT = (
    "You are a support assistant answering questions about company policy documents.\n"
    "Rules:\n"
    "1. Answer ONLY using the numbered context passages provided.\n"
    f"2. If the answer is NOT in the passages, reply with exactly: {REFUSAL_SENTINEL}\n"
    "3. Otherwise answer concisely and cite passage numbers like [1], [2].\n"
    "Do not use outside knowledge. Do not guess. Do not invent details."
)

def answer_with_refusal(question, k=5, threshold=SCORE_THRESHOLD):
    hits = retrieve(question, k=k)
    top_score = hits[0]["score"] if hits else 0.0

    # Layer 1: cheap score-threshold gate (no LLM call if retrieval is weak)
    if not hits or top_score < threshold:
        return {
            "status": "refused-low-confidence",
            "answer": "I don't know based on the documents.",
            "top_score": top_score,
            "sources": [],
        }

    # Layer 2: strict prompt — model must use the sentinel to refuse
    context = "\n\n".join(
        f"[{i+1}] (source: {h['source']})\n{h['content']}" for i, h in enumerate(hits)
    )
    resp = aoai.chat.completions.create(
        model=CHAT_DEPLOYMENT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context passages:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0,
    )
    raw = resp.choices[0].message.content.strip()

    if raw.startswith(REFUSAL_SENTINEL):
        return {
            "status": "refused-by-model",
            "answer": "I don't know based on the documents.",
            "top_score": top_score,
            "sources": [],
        }

    return {
        "status": "answered",
        "answer": raw,
        "top_score": top_score,
        "sources": sorted({h["source"] for h in hits}),
    }

## 6. Quick demo

In [8]:
demo = [
    "How many PTO days do full-time employees get?",
    "Can I work remotely, and are there any rules?",
    "What is the company's stock price?",          # out of scope
    "How do I set up a Salesforce account?",       # out of scope
]
for q in demo:
    r = answer_with_refusal(q)
    print(f"Q: {q}")
    print(f"   status = {r['status']}   top_score = {r['top_score']:.3f}")
    print(f"   A: {r['answer']}")
    if r["sources"]:
        print(f"   sources: {r['sources']}")
    print("-" * 70)

Q: How many PTO days do full-time employees get?
   status = answered   top_score = 0.724
   A: Full-time employees accrue 15 days of paid time off (PTO) per calendar year. After five years of continuous service, the annual PTO allowance increases to 20 days per year [2].
   sources: ['01_PTO_and_Leave_Policy.pdf', '02_New_Hire_Onboarding_Guide.pdf', '03_Remote_Work_Policy.pdf', '04_Employee_Benefits_Overview.pdf']
----------------------------------------------------------------------
Q: Can I work remotely, and are there any rules?
   status = answered   top_score = 0.675
   A: Yes, you can work remotely if your role is eligible and you have prior manager approval. Remote work is allowed up to 3 days per week, with at least 2 days on-site unless a fully remote arrangement is approved by a department head. While working remotely, you must be available during core hours of 10:00 am to 3:00 pm local time and attend scheduled meetings via video. Additionally, you must use a company-provid

## 7. The evaluation set

Each question is tagged `answerable=True`.If the answer is in your indexed PDFs, otherwise `False`. This is the ground truth for the **decision** to answer vs refuse.

In [9]:
EVAL = [
    # In-scope: answer IS in the indexed PDFs
    {"q": "How many PTO days do full-time employees get?",          "answerable": True},
    {"q": "When do new hires start accruing PTO?",                  "answerable": True},
    {"q": "Can I work remotely, and are there any rules?",          "answerable": True},
    {"q": "How many days per week can I work remotely?",            "answerable": True},
    {"q": "What is the 401k match?",                                "answerable": True},
    {"q": "How many paid holidays are there per year?",             "answerable": True},
    {"q": "How many sick days do full-time employees get?",         "answerable": True},
    # Out-of-scope: answer is NOT in the documents
    {"q": "What is the company's stock price?",                     "answerable": False},
    {"q": "How do I file a patent?",                                "answerable": False},
    {"q": "Who is the CEO of Acme Corp?",                           "answerable": False},
    {"q": "What is the office WiFi password?",                      "answerable": False},
    {"q": "How do I set up a Salesforce account?",                  "answerable": False},
    {"q": "What was the company's revenue last quarter?",           "answerable": False},
]
print(f"{sum(1 for e in EVAL if e['answerable'])} in-scope, {sum(1 for e in EVAL if not e['answerable'])} out-of-scope.")

7 in-scope, 6 out-of-scope.


## 8. Run the eval — confusion matrix

We classify each result against ground truth:

- **TP** — in-scope and we answered (good)
- **FN** — in-scope but we refused (over-cautious — too few useful answers)
- **TN** — out-of-scope and we refused (good — the whole point of today)
- **FP** — out-of-scope but we answered (**hallucination risk** — the worst failure mode)

The signal that matters most for an FAQ bot is **low FP** — never answering questions that aren't grounded in the documents.

In [10]:
results = []
for item in EVAL:
    r = answer_with_refusal(item["q"])
    results.append({
        **item,
        "answered": r["status"] == "answered",
        "top_score": r["top_score"],
        "status": r["status"],
        "answer": r["answer"],
    })

TP = sum(1 for r in results if r["answerable"]      and r["answered"])
FN = sum(1 for r in results if r["answerable"]      and not r["answered"])
TN = sum(1 for r in results if not r["answerable"]  and not r["answered"])
FP = sum(1 for r in results if not r["answerable"]  and r["answered"])

print(f"In-scope answered (TP):     {TP}/{TP+FN}")
print(f"In-scope refused  (FN):     {FN}/{TP+FN}   <- over-cautious")
print(f"Out-of-scope refused (TN):  {TN}/{TN+FP}")
print(f"Out-of-scope answered (FP): {FP}/{TN+FP}   <- HALLUCINATION RISK")
print()
print(f"{'Question':<55} {'expected':<8} {'actual':<8} {'score':<6}  status")
print("-" * 105)
for r in results:
    exp = "answer" if r["answerable"] else "refuse"
    act = "answer" if r["answered"]   else "refuse"
    mark = "  " if (exp == act) else " *"
    print(f"{r['q'][:53]:<55} {exp:<8} {act:<8} {r['top_score']:.3f} {mark}{r['status']}")

In-scope answered (TP):     5/7
In-scope refused  (FN):     2/7   <- over-cautious
Out-of-scope refused (TN):  6/6
Out-of-scope answered (FP): 0/6   <- HALLUCINATION RISK

Question                                                expected actual   score   status
---------------------------------------------------------------------------------------------------------
How many PTO days do full-time employees get?           answer   answer   0.724   answered
When do new hires start accruing PTO?                   answer   answer   0.737   answered
Can I work remotely, and are there any rules?           answer   answer   0.675   answered
How many days per week can I work remotely?             answer   answer   0.713   answered
What is the 401k match?                                 answer   refuse   0.589  *refused-low-confidence
How many paid holidays are there per year?              answer   refuse   0.625  *refused-low-confidence
How many sick days do full-time employees get?          ans

## 9. Threshold sweep — pick the cutoff from your own data

A single threshold value picked out of thin air is fragile. Below, we sweep `SCORE_THRESHOLD` across a range and see how the confusion matrix shifts. Move the threshold up to refuse more aggressively (reduces FP at the cost of FN); move it down to answer more readily (the opposite).

To avoid re-calling the LLM at every threshold, we run each eval question once with `threshold=0.0` (forcing the LLM to decide), record the LLM's verdict and the top score, then re-apply the gate at different thresholds purely with arithmetic.

In [11]:
# 1) One pass with threshold disabled, to record LLM verdict + top score per question
records = []
for item in EVAL:
    r = answer_with_refusal(item["q"], threshold=0.0)
    records.append({
        **item,
        "top_score":    r["top_score"],
        "llm_answered": r["status"] == "answered",
    })

# 2) Re-apply different gates without re-calling anything
print(f"{'threshold':<11} {'TP':<4} {'FN':<4} {'TN':<4} {'FP':<4}")
print("-" * 35)
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    tp = fn = tn = fp = 0
    for r in records:
        refused_by_gate = r["top_score"] < t
        final_answered = (not refused_by_gate) and r["llm_answered"]
        if   r["answerable"] and final_answered:         tp += 1
        elif r["answerable"] and not final_answered:     fn += 1
        elif not r["answerable"] and not final_answered: tn += 1
        else:                                            fp += 1
    print(f"{t:<11.2f} {tp:<4} {fn:<4} {tn:<4} {fp:<4}")

print()
print("Score distribution:")
for r in sorted(records, key=lambda x: x["top_score"], reverse=True):
    tag = "in-scope    " if r["answerable"] else "out-of-scope"
    print(f"  {r['top_score']:.3f}  {tag}  {r['q'][:60]}")

threshold   TP   FN   TN   FP  
-----------------------------------
0.50        7    0    6    0   
0.55        7    0    6    0   
0.60        6    1    6    0   
0.65        5    2    6    0   
0.70        3    4    6    0   
0.75        0    7    6    0   
0.80        0    7    6    0   
0.85        0    7    6    0   

Score distribution:
  0.737  in-scope      When do new hires start accruing PTO?
  0.724  in-scope      How many PTO days do full-time employees get?
  0.713  in-scope      How many days per week can I work remotely?
  0.675  in-scope      Can I work remotely, and are there any rules?
  0.665  out-of-scope  Who is the CEO of Acme Corp?
  0.661  in-scope      How many sick days do full-time employees get?
  0.625  in-scope      How many paid holidays are there per year?
  0.589  in-scope      What is the 401k match?
  0.581  out-of-scope  What is the office WiFi password?
  0.579  out-of-scope  How do I set up a Salesforce account?
  0.558  out-of-scope  What is the c

## Day 3 checklist

1. All in-scope questions answered (TP equals in-scope count)
2. All out-of-scope questions refused (FP = 0)
3. Threshold tuned from the sweep, not guessed
4. You can articulate when Layer 1 (score gate) fires vs Layer 2 (model sentinel)

### Learnings:
1. The whole reason RAG is trusted in business is that it can say "I don't know" instead of inventing.
2. the score gate refuses without paying for an LLM call.
3. a confusion matrix tells you whether the bot is safe to deploy.

### Next up — Chat with messy data + metadata filtering
Swap the clean handbook PDFs for messier mixed content (notes, markdown, dated files), attach metadata during ingestion (`date`, `source`, `category`), and add **filtered retrieval** so a query like "only notes from 2024" applies a filter alongside the vector search.